# 02 — Data Preprocessing (Week 3)

**Goal:** turn the raw CRMLS *sold* data into a clean, model-ready dataset.

Steps in this notebook:
1. **Load & combine** the monthly files (reuse Week 2 logic), restrict to Single-Family Residential.
2. **Select features** — keep predictive columns, drop IDs, agent info, and anything that leaks the target.
3. **Handle outliers** — remove non-market / impossible records.
4. **Handle missing values** — drop rows with no target, decide impute vs. flag for the rest.
5. **Encode categoricals** — turn text fields into numbers.
6. **Scale numeric features** — needed for linear models (not for trees).
7. **Time-based train/test split** — most recent month = test, the *X* months before it = train, **X is tunable**.
8. **Save** the cleaned CSV.

> Note on leakage: scaling and any target-based encoding are **fit on the training set only**, inside the
> split function (step 7), so information from the test month never bleeds into training.

In [1]:
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Load & combine monthly files

Same approach as `01_exploration.ipynb`: read each monthly CSV, keep the **common columns** across files
(schemas differ between `_filled` and raw months), concatenate, and tag each row with its `year_month`.

In [2]:
DATA_DIR = "data/sold"
PATTERN = "CRMLSSold202[456]*.csv"   # 2024, 2025, 2026 monthly files. Adjust to widen the window.

files = sorted(glob.glob(f"{DATA_DIR}/{PATTERN}"))
print(f"Matched {len(files)} files")

dfs = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    ym = re.search(r"(\d{6})", f.split("/")[-1])          # grab YYYYMM from file name
    d["year_month"] = ym.group(1) if ym else "unknown"
    dfs.append(d)

common = set.intersection(*[set(d.columns) for d in dfs])
df = pd.concat([d[list(common)] for d in dfs], ignore_index=True)
print(f"Combined: {df.shape[0]:,} rows x {df.shape[1]} cols")

Matched 29 files
Combined: 636,443 rows x 77 cols


In [3]:
# Restrict to Single-Family Residential (per the task prompt)
before = len(df)
df = df[(df["PropertyType"] == "Residential") &
        (df["PropertySubType"] == "SingleFamilyResidence")].copy()
print(f"SFR filter: {before:,} -> {len(df):,} rows ({len(df)/before*100:.1f}% kept)")

SFR filter: 636,443 -> 320,506 rows (50.4% kept)


## 2. Select features

We keep columns that plausibly influence sale price and are **knowable for any property** (whether or not
it is currently for sale).

**Deliberately dropped as target leakage** — a property that is *not for sale* has no list price, and these
are only known *after* the sale, so they can't be inputs to a general price predictor:
`ListPrice`, `OriginalListPrice`, `DaysOnMarket`, `CloseDate`-derived price stats, buyer/seller info.

**Dropped as non-predictive** — IDs, agent/office names, free-text address fields.

In [4]:
TARGET = "ClosePrice"

# Numeric predictors (physical characteristics of the home + location coordinates)
NUMERIC_FEATURES = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "GarageSpaces",
    "Stories", "FireplacesTotal", "Latitude", "Longitude",
]

# Categorical predictors (location / type). PropertySubType is constant after the filter, so it's excluded.
CATEGORICAL_FEATURES = ["CountyOrParish", "City", "PostalCode"]

# Kept only to build the time-based split (not a model feature itself)
KEEP_FOR_SPLIT = ["CloseDate", "year_month"]

keep = [TARGET] + NUMERIC_FEATURES + CATEGORICAL_FEATURES + KEEP_FOR_SPLIT
keep = [c for c in keep if c in df.columns]      # guard against a missing column
missing_cols = set([TARGET] + NUMERIC_FEATURES + CATEGORICAL_FEATURES) - set(df.columns)
if missing_cols:
    print("WARNING: expected columns not found:", missing_cols)

data = df[keep].copy()

# Coerce numerics (some come in as object due to stray text)
for c in NUMERIC_FEATURES + [TARGET]:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data["CloseDate"] = pd.to_datetime(data["CloseDate"], errors="coerce")
print(data.shape)
data.head()

(320506, 16)


,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,Stories,FireplacesTotal,Latitude,Longitude,CountyOrParish,City,PostalCode,CloseDate,year_month
5,815000.0,1974.0,4.0,4.0,NaN,2023.0,2.0,2.0,NaN,32.659315,-117.096922,San Diego,National City,91950,2024-01-05,202401
9,810000.0,1974.0,4.0,4.0,NaN,2023.0,2.0,2.0,NaN,32.659284,-117.097174,San Diego,National City,91950,2024-01-05,202401
28,2100000.0,3736.0,4.0,4.0,11219.0,2019.0,3.0,2.0,NaN,35.277199,-120.646786,San Luis Obispo,San Luis Obispo,93401,2024-01-02,202401
29,1950000.0,2100.0,3.0,0.0,74487.6,1986.0,2.0,2.0,NaN,39.419080,-123.814842,Mendocino,Fort Bragg,95437,2024-01-22,202401
31,2340000.0,2442.0,4.0,3.0,8712.0,1985.0,3.0,NaN,NaN,37.325454,-121.780303,Santa Clara,San Jose,95148,2024-01-31,202401


## 3. Handle outliers

Domain rules to strip out non-market transactions (e.g. \$1 intra-family transfers) and impossible values.
Bounds are intentionally generous — the aim is to remove garbage, not to trim legitimate high-end homes.

In [5]:
before = len(data)

rules = {
    "ClosePrice out of [10k, 50M]":      (data.ClosePrice < 1e4) | (data.ClosePrice > 5e7),
    "LivingArea out of [200, 20000]":    (data.LivingArea < 200) | (data.LivingArea > 20000),
    "Bedrooms out of [0, 12]":           (data.BedroomsTotal < 0) | (data.BedroomsTotal > 12),
    "Bathrooms out of [0, 12]":          (data.BathroomsTotalInteger < 0) | (data.BathroomsTotalInteger > 12),
    "LotSize > 20 acres (871200 sqft)":  (data.LotSizeSquareFeet > 871200),
    "YearBuilt out of [1850, 2026]":     (data.YearBuilt < 1850) | (data.YearBuilt > 2026),
}

for label, mask in rules.items():
    print(f"  {label:38s}: {int(mask.fillna(False).sum()):>7,} flagged")

drop_mask = np.zeros(len(data), dtype=bool)
for mask in rules.values():
    drop_mask |= mask.fillna(False).values

data = data[~drop_mask].copy()
print(f"\nOutlier removal: {before:,} -> {len(data):,} rows ({(before-len(data))/before*100:.1f}% dropped)")

  ClosePrice out of [10k, 50M]          :      43 flagged
  LivingArea out of [200, 20000]        :     159 flagged
  Bedrooms out of [0, 12]               :      21 flagged
  Bathrooms out of [0, 12]              :      71 flagged
  LotSize > 20 acres (871200 sqft)      :     931 flagged
  YearBuilt out of [1850, 2026]         :       9 flagged

Outlier removal: 320,506 -> 319,313 rows (0.4% dropped)


## 4. Handle missing values

- **Target (`ClosePrice`):** rows with a missing target are useless for supervised training → **drop**.
- **Numeric features:** median-impute. We fit the median on the **training set only** (done in step 7) to
  avoid leakage, so here we just report the null rates and add missing-flag columns for the worst offenders.
- **Categoricals:** fill with the literal string `"Unknown"` so they become their own category.

In [6]:
# Drop rows with no target
before = len(data)
data = data[data[TARGET].notna()].copy()
print(f"Dropped {before - len(data):,} rows with missing target")

# Null rate report
null_rate = (data[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().mean() * 100).round(1)
print("\nNull rate (%) per feature:")
print(null_rate.sort_values(ascending=False))

# Add missing-flags for features with meaningful missingness (>2%); the flag can itself be predictive.
FLAG_THRESHOLD = 2.0
flag_cols = null_rate[null_rate > FLAG_THRESHOLD].index.intersection(NUMERIC_FEATURES)
for c in flag_cols:
    data[f"{c}_missing"] = data[c].isna().astype(int)
print("\nAdded missing-flags for:", list(flag_cols))

# Categoricals -> fill 'Unknown'
for c in CATEGORICAL_FEATURES:
    data[c] = data[c].astype("string").fillna("Unknown")

Dropped 2 rows with missing target

Null rate (%) per feature:
FireplacesTotal          100.0
Stories                   12.7
GarageSpaces               3.7
LotSizeSquareFeet          1.7
LivingArea                 0.1
YearBuilt                  0.1
City                       0.1
BedroomsTotal              0.0
BathroomsTotalInteger      0.0
Latitude                   0.0
Longitude                  0.0
CountyOrParish             0.0
PostalCode                 0.0
dtype: float64

Added missing-flags for: ['GarageSpaces', 'Stories', 'FireplacesTotal']


## 5. Encode categoricals

`CountyOrParish`, `City`, and `PostalCode` are **high-cardinality** location fields (hundreds of unique
values), so one-hot encoding would explode the column count. We use **frequency encoding** (map each
category to how often it appears) as a simple, leak-safe numeric encoding.

> A stronger option for Week 6 is **target encoding** (mean ClosePrice per category), but that must be fit
> on the training set only. We keep the raw string columns in the cleaned CSV so later weeks can re-encode.

In [7]:
for c in CATEGORICAL_FEATURES:
    print(f"  {c}: {data[c].nunique():,} unique values")

# Frequency encoding (computed on the full cleaned set here for the CSV; for strict modeling,
# recompute on train only — see the split function in step 7).
for c in CATEGORICAL_FEATURES:
    freq = data[c].value_counts(normalize=True)
    data[f"{c}_freq"] = data[c].map(freq)

  CountyOrParish: 63 unique values
  City: 1,104 unique values
  PostalCode: 3,225 unique values


## 6. Feature scaling

Linear models (Week 4 baseline, Ridge/Lasso) need standardized inputs; tree/boosting models don't.
Scaling **must be fit on the training set only**, so we don't scale here — we build a reusable
`StandardScaler` inside the split function below and expose both scaled and unscaled feature matrices.

## 7. Time-based train/test split (X is tunable)

Per the task doc: **the most recent month is the test set**, and the **X months immediately before it are
the training set**. `X` is *not* fixed — it's a hyperparameter. The function below builds a leakage-safe
split for a given `X`:

- imputes numeric medians from **train only**,
- fits frequency encoding on **train only**,
- fits the scaler on **train only**,
- returns both scaled (for linear models) and unscaled (for trees) matrices.

Use it in Week 4+ to sweep `X` and pick the value that maximizes test R².

In [8]:
# Ordered list of available months
data["month_key"] = data["CloseDate"].dt.to_period("M")
months = sorted(data["month_key"].dropna().unique())
print("Available months:", [str(m) for m in months])

MODEL_NUMERIC = NUMERIC_FEATURES + [c for c in data.columns if c.endswith("_missing")]
MODEL_CATEGORICAL_ENC = [f"{c}_freq" for c in CATEGORICAL_FEATURES]


def build_train_test(X, data=data, scale=True):
    """Return a leakage-safe time-based split.

    test  = most recent month
    train = the X months immediately before it
    Returns dict with X_train, X_test (scaled), X_train_raw, X_test_raw (unscaled), y_train, y_test.
    """
    from sklearn.preprocessing import StandardScaler

    test_month = months[-1]
    train_months = months[-(X + 1):-1]          # X months right before the test month
    if len(train_months) < X:
        raise ValueError(f"Only {len(months)-1} months available before the test month; X={X} too large.")

    train = data[data.month_key.isin(train_months)].copy()
    test = data[data.month_key == test_month].copy()

    # Impute numeric medians from TRAIN only
    medians = train[NUMERIC_FEATURES].median()
    train[NUMERIC_FEATURES] = train[NUMERIC_FEATURES].fillna(medians)
    test[NUMERIC_FEATURES] = test[NUMERIC_FEATURES].fillna(medians)

    # Frequency-encode categoricals from TRAIN only (unseen categories -> 0)
    for c in CATEGORICAL_FEATURES:
        freq = train[c].value_counts(normalize=True)
        train[f"{c}_freq"] = train[c].map(freq).fillna(0.0)
        test[f"{c}_freq"] = test[c].map(freq).fillna(0.0)

    feat_cols = MODEL_NUMERIC + MODEL_CATEGORICAL_ENC
    X_train_raw, X_test_raw = train[feat_cols], test[feat_cols]
    y_train, y_test = train[TARGET], test[TARGET]

    out = {"feature_cols": feat_cols,
           "X_train_raw": X_train_raw, "X_test_raw": X_test_raw,
           "y_train": y_train, "y_test": y_test,
           "test_month": str(test_month), "train_months": [str(m) for m in train_months]}

    if scale:
        scaler = StandardScaler().fit(X_train_raw)
        out["X_train"] = pd.DataFrame(scaler.transform(X_train_raw), columns=feat_cols, index=X_train_raw.index)
        out["X_test"] = pd.DataFrame(scaler.transform(X_test_raw), columns=feat_cols, index=X_test_raw.index)
        out["scaler"] = scaler
    return out


# Demo with X = 6
split = build_train_test(X=6)
print("\ntest month:", split["test_month"])
print("train months:", split["train_months"])
print("train rows:", len(split["y_train"]), "| test rows:", len(split["y_test"]))
print("feature cols:", split["feature_cols"])

Available months: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']

test month: 2026-05
train months: ['2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
train rows: 59169 | test rows: 11976
feature cols: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Stories', 'FireplacesTotal', 'Latitude', 'Longitude', 'GarageSpaces_missing', 'Stories_missing', 'FireplacesTotal_missing', 'CountyOrParish_freq', 'City_freq', 'PostalCode_freq']


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


## 8. Save the cleaned CSV

Deliverable for Week 3. We save the **cleaned, pre-split** dataset (SFR-filtered, outliers removed, target
present, categoricals filled, encodings + missing-flags added). The train/test split is intentionally *not*
baked in — Week 4 calls `build_train_test(X)` to sweep the window length.

In [9]:
import os
os.makedirs("data/processed", exist_ok=True)
out_path = "data/processed/sold_clean.csv"
data.drop(columns=["month_key"]).to_csv(out_path, index=False)
print(f"Saved {len(data):,} rows x {data.shape[1]-1} cols -> {out_path}")

Saved 319,311 rows x 22 cols -> data/processed/sold_clean.csv
